# Фаза 0: обучение на парах `edit_file` + замер на 20 задачах (Colab A100 80 ГБ)

Цель: проверить, лечит ли обучение на парах «хороший / плохой вызов `edit_file`» литеральный `\n`
в `old_string`/`new_string` (на базе 28%, у `sft_think_v1` 48%). Критерий G1: литеральный `\n` < 10% вызовов
и решено >= 7 из 20.

Порядок: **1** проверка карты, **2** код и зависимости, **3** Drive, **4** модель, **5** данные,
**6** обучение (~2,5 ч, vLLM НЕ запускать), **7** сохранение в Drive, затем замер: **8** vLLM с адаптером,
**9** сервер песочницы с ноутбука, **10** прогон 20 задач, **11** анализ.
Если среда перезапустилась, выполните ячейки 1-4 заново; адаптер берётся из Drive.

## 1. Карта (нужна A100 80 ГБ; на 40 ГБ и L4 обучение не поместится)

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
print(out)
name, total, used = [x.strip() for x in out.split(",")]
assert int(total) >= 79000, f"нужна карта на 80 ГБ, а здесь {total} МиБ: смените среду выполнения"
assert int(used) < 2000, f"занято {used} МиБ: остановите vLLM (ячейка в конце ноутбука) или перезапустите среду"

## 2. Код и зависимости

In [ ]:
import os
if not os.path.exists("/content/gemma4-swe-rl"):
    !git clone -q https://github.com/aletonfdg/gemma4-swe-rl.git /content/gemma4-swe-rl
%cd /content/gemma4-swe-rl
!git pull -q
!git log --oneline | head -3
# torch и transformers в Colab уже есть; не обновляем их, чтобы не сломать совместимость
!pip install -q compressed-tensors bitsandbytes accelerate safetensors huggingface_hub kagglehub openai pyyaml requests
import torch, transformers
print("torch", torch.__version__, "transformers", transformers.__version__)

## 3. Google Drive (адаптеры и логи храним здесь: папки Colab при перезапуске пропадают)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/gemma_adapters"
os.makedirs(DRIVE, exist_ok=True)
!ls -la {DRIVE}

## 4. Базовая модель в `/content/model`
Обучение берёт её отсюда (`GEMMA_MODEL_DIR`), vLLM позже тоже. Если папка уже есть, ячейка ничего не качает.
Нужна свежая версия кода (`git pull` в ячейке 2): без неё обучение скачает модель из Hugging Face ещё раз.

In [ ]:
import getpass, os
MODEL_DIR = "/content/model"
HF_REPO = "google/gemma-4-31b-it-qat-w4a16-ct"
KAGGLE_MODEL = "google/gemma-4/transformers/gemma-4-31b-it-qat-w4a16-ct"

if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    try:
        from huggingface_hub import snapshot_download
        hf_token = getpass.getpass("HF токен (Enter, если не нужен): ").strip() or None
        snapshot_download(HF_REPO, local_dir=MODEL_DIR, token=hf_token)
        print("Скачано с Hugging Face")
    except Exception as e:
        print("HF не получилось:", type(e).__name__, str(e)[:300])
        import kagglehub
        os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("KAGGLE_API_TOKEN (KGAT_...): ").strip()
        path = kagglehub.model_download(KAGGLE_MODEL)
        !cp -r "{path}"/. {MODEL_DIR}/
        print("Скачано с Kaggle Models")
os.environ["GEMMA_MODEL_DIR"] = MODEL_DIR
!ls {MODEL_DIR} | head -20; du -sh {MODEL_DIR}

## 5. Данные: `data/pref_edit.jsonl` (188 пар; папка `data/` в git не попадает)
Либо положите файл в Drive (`gemma_adapters/pref_edit.jsonl`) и он подтянется сам, либо загрузите вручную
(`C:\Users\vishn\Desktop\gemma4-swe-rl\data\pref_edit.jsonl`); при ручной загрузке копия уйдёт в Drive.

In [ ]:
import shutil
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/pref_edit.jsonl"):
    if os.path.exists(f"{DRIVE}/pref_edit.jsonl"):
        shutil.copy(f"{DRIVE}/pref_edit.jsonl", "data/pref_edit.jsonl")
    else:
        from google.colab import files
        files.upload()
        os.replace("pref_edit.jsonl", "data/pref_edit.jsonl")
        shutil.copy("data/pref_edit.jsonl", f"{DRIVE}/pref_edit.jsonl")
n = sum(1 for l in open("data/pref_edit.jsonl", encoding="utf-8") if l.strip())
print("пар в файле:", n)
assert n == 188, "ожидалось 188 пар"

## 6. Обучение (~2,5 ч)
Смотрите в логе: `CompressedLinear: 0` (это нормально при `--decompress`), `пар: ~177` (отброшено <= ~15),
`eval до/в конце` (доля chosen>rejected и разность logp), `pref-loss` около 0.69 в начале и вниз.
При `CUDA out of memory` добавьте `--max-len 6144`. Обучение идёт отдельным процессом: после него память
освобождается сама. Лог пишется в `/content/pref_train.log`.

In [ ]:
!python -m train.pref_lora --data data/pref_edit.jsonl --name pref_edit_v1 --decompress --epochs 2 2>&1 | tee /content/pref_train.log

## 7. Сохранить адаптер и лог в Drive (сразу после обучения)

In [ ]:
!cp -r adapters/pref_edit_v1 {DRIVE}/ && cp /content/pref_train.log {DRIVE}/pref_train.log
!ls -la {DRIVE}/pref_edit_v1
!grep -E "пар:|eval|CompressedLinear" /content/pref_train.log

## 8. vLLM с адаптером
Если среда не перезапускалась, можно сразу запускать: после обучения память свободна. После перезапуска сначала
выполните ячейки 1-4; адаптер скопируется из Drive.

In [ ]:
import os, subprocess, time, requests
os.chdir("/content/gemma4-swe-rl")
DRIVE = "/content/drive/MyDrive/gemma_adapters"
ADAPTER = "pref_edit_v1"
os.makedirs("adapters", exist_ok=True)
if not os.path.exists(f"adapters/{ADAPTER}"):
    !cp -r {DRIVE}/{ADAPTER} adapters/
!pip install -q vllm
out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
assert int(out) < 2000, f"занято {out} МиБ: остановите лишние процессы или перезапустите среду"
cmd = ["vllm", "serve", "/content/model", "--served-model-name", "gemma-4-31b-it-qat-w4a16-ct",
       "--quantization", "compressed-tensors", "--dtype", "bfloat16",
       "--max-model-len", "32768", "--gpu-memory-utilization", "0.92", "--max-num-seqs", "4",
       "--enable-lora", "--max-lora-rank", "64", "--max-loras", "1",
       "--lora-modules", f"{ADAPTER}=/content/gemma4-swe-rl/adapters/{ADAPTER}",
       "--enable-auto-tool-choice", "--tool-call-parser", "gemma4", "--reasoning-parser", "gemma4",
       "--default-chat-template-kwargs", '{"enable_thinking": true}', "--port", "8000"]
log = open("/content/vllm.log", "w")
proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
t0 = time.time()
while time.time() - t0 < 1500:
    if proc.poll() is not None:
        break
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=3).status_code == 200:
            print("vLLM готов за %.0f c" % (time.time() - t0)); break
    except requests.RequestException:
        pass
    time.sleep(5)
if proc.poll() is not None:
    print("vLLM упал, конец лога:"); print(open("/content/vllm.log").read()[-3000:])
print([m["id"] for m in requests.get("http://127.0.0.1:8000/v1/models").json()["data"]])

## 9. Сервер песочницы (на ноутбуке)
В двух окнах PowerShell из папки проекта (Docker Desktop должен работать, ноутбук не должен засыпать):

```
python -m sim.mimo_server --data C:\Users\vishn\Desktop\data\mimo_v2_6_code --max-sessions 4 --ttl 2400
cloudflared tunnel --url http://127.0.0.1:8765
```
Сервер печатает токен, туннель печатает адрес `https://....trycloudflare.com`. Введите их ниже.
Образы Docker сохраняются (флаг `--prune-images` здесь не нужен: для 20 задач место есть).

In [ ]:
import getpass, json, urllib.request
os.environ["SWE_SANDBOX_URL"] = input("Адрес туннеля: ").strip().rstrip("/")
os.environ["SWE_SANDBOX_TOKEN"] = getpass.getpass("Токен сервера: ").strip()
print(urllib.request.urlopen(os.environ["SWE_SANDBOX_URL"] + "/health", timeout=30).read())

## 10. Прогон тех же 20 задач
Те же задачи и настройки, что у `sft_think_v1_eval20` (температура 0.7, мышление, системный промпт по умолчанию),
чтобы сравнение было честным. Около часа. Результаты: `results/pref_edit_v1_eval20`.

In [ ]:
IDS = """
fastapi_14303 fastapi_14786 fastapi_14361 requests_7315 rich_2725 rich_3882 fastapi_12942 fastapi_14356
fastapi_14246 fastapi_14258 fastapi_14306 fastapi_14371 fastapi_14479 fastapi_14487 fastapi_14512 fastapi_14605
rich_3052 rich_3180 rich_3486 rich_3930
""".split()
assert len(IDS) == 20 and len(set(IDS)) == 20, len(IDS)
run_cmd = " ".join(["python -m sim.run_mimo_baseline --model", ADAPTER, "--temperature 0.7 --repeats 1 --workers 2",
                    "--thinking on --out results/pref_edit_v1_eval20 --ids"] + IDS)
!{run_cmd}

## 11. Анализ (критерий G1: литеральный `\n` < 10% и решено >= 7 из 20)

In [ ]:
!python -m tools.analyze_edits results/pref_edit_v1_eval20
import json
last = {}
for l in open("results/pref_edit_v1_eval20/results.jsonl", encoding="utf-8"):
    r = json.loads(l)
    last[(r["instance_id"], r.get("rep", 0))] = r
print("решено:", sum(1 for r in last.values() if r["passed"]), "из", len(last))
!zip -qr {DRIVE}/pref_edit_v1_eval20.zip results/pref_edit_v1_eval20 && echo "копия результатов в Drive"

## Приложение: остановить vLLM
```
!pkill -f vllm
```
Если память не освободилась, перезапустите среду выполнения.